This explains how stochastic gradient descent (SGD) works inside neural networks by tracking loss after every mini-batch instead of just after full epochs.



### Epoch vs. Batch Loss Tracking

Default Keras behavior:
- Keras shows loss after each epoch (one full pass through all data).
- With 150 data points and default batch size = 32, each epoch has 5 mini-batches (150 ÷ 32 ≈ 5).

Epoch process:
```
Epoch 1:
1. Batch 1 (points 0-31) → compute gradient → update weights
2. Batch 2 (points 32-63) → compute gradient → update weights  
3. Batch 3 (points 64-95) → compute gradient → update weights
4. Batch 4 (points 96-127) → compute gradient → update weights
5. Batch 5 (points 128-149) → compute gradient → update weights
→ NOW record epoch loss (average across all 5 batches)
```

The smooth epoch loss curve is actually every 5th point from the detailed batch-by-batch loss.



### Batch Loss is Noisier

When you track loss after every batch (using custom callbacks):

- You see 2,500 total batches over 500 epochs (500 × 5 batches/epoch).
- The curve is much noisier because each gradient is computed on only 32 data points (rough approximation).
- Epoch losses = `batch_losses[4::5]` (every 5th batch) → smooth curve.



### Batch Size Effects

#### Batch size = 32 (default)
```
150 data points ÷ 32 = 5 gradient steps per epoch
```
- Gradient quality: Good approximation (32 points average the noise).
- Speed per step: Slower (compute gradients for 32 points).
- Total steps per epoch: 5.

#### Batch size = 1 (pure SGD)
```
150 data points ÷ 1 = 150 gradient steps per epoch
```
- Gradient quality: Very noisy (single point = poor approximation).
- Speed per step: Very fast (just 1 point).
- Total steps per epoch: 150.



### The Counterintuitive Result

Plotting loss vs EPOCHS (not vs batches) reveals a confusing pattern:

- Batch size 1 appears to converge faster per epoch.
- Batch size 32 appears slower per epoch.

Why this happens:
```
Batch size 1:   150 gradient steps per epoch = rapid progress per epoch
Batch size 32:   5 gradient steps per epoch = slower progress per epoch
```

But total computational work is different:
```
After 1 epoch:
• Batch 1: 150 gradient computations  
• Batch 32: 5 gradient computations (160 total point-gradients)
```

The batch size 1 model took 30x more gradient computations per epoch, so it naturally progressed faster per epoch, even though each individual gradient step was lower quality.



### Core Tradeoff (Batch Size)

Batch size controls:

```
Smaller batch size (→ 1)        Larger batch size (→ 32)
├── Noisier gradients           ├── Smoother gradients
├── Faster per step             ├── Slower per step  
├── More steps per epoch       ├── Fewer steps per epoch
└── Often slower total convergence  └── Often faster total convergence
```

Key insight: When comparing batch sizes, always compare loss vs total gradient steps (batches), not vs epochs.



### Random Shuffling

Real SGD shuffles data before each epoch:
```
Epoch 1: [random order of all 150 points]
Epoch 2: [NEW random order of all 150 points] 
```
Sequential batches (1-32, 33-64, etc.) don't work well. Random ordering ensures each mini-batch is a fresh sample.



Summary: Batch-level loss tracking reveals the noisy reality of SGD. Smaller batches = noisier but more frequent updates. Always compare models by total computational steps, not just epochs.